In [ ]:
"""
"""

import pandas as pd
import numpy as np
import joblib
import pickle

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier




class Config:
    DATA_PATH = Path("german_credit_data.csv")
    TEST_SIZE = 0.2
    RANDOM_STATE = 42
    CV_SPLITS = 5

    MODEL_PATH = "model.pkl"
    SCALER_PATH = "scaler.pkl"
    ENCODER_PATH = "encoder.pkl"
    MERGED_DATA_PATH = "merged_data.csv"




class DataProcessor:
    def __init__(self, path):
        self.path = path
        self.encoder = None
        self.scaler = None
        self.cat_cols = None

    def load_data(self):
        df = pd.read_csv(self.path)

        df["Saving accounts"].fillna("unknown", inplace=True)
        df["Checking account"].fillna("unknown", inplace=True)

        return df

    def preprocess(self, df):
        X_raw = df.drop(["Risk", "Index"], axis=1)
        y = df["Risk"]

        self.cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns

        # Encoding
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded = self.encoder.fit_transform(X_raw[self.cat_cols])

        encoded_df = pd.DataFrame(
            encoded,
            columns=self.encoder.get_feature_names_out(self.cat_cols)
        )

        X = pd.concat(
            [X_raw.drop(self.cat_cols, axis=1).reset_index(drop=True), encoded_df],
            axis=1
        )

        # Scaling
        self.scaler = StandardScaler()
        X = pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)

        return X, y



class ModelFactory:
    @staticmethod
    def get_models():
        return {
            "Random Forest": {
                "model": RandomForestClassifier(random_state=Config.RANDOM_STATE),
                "params": {
                    "n_estimators": [100, 200],
                    "max_depth": [None, 10],
                }
            },
            "Gradient Boosting": {
                "model": GradientBoostingClassifier(random_state=Config.RANDOM_STATE),
                "params": {
                    "n_estimators": [100, 200],
                    "learning_rate": [0.01, 0.1],
                }
            },
            "Logistic Regression": {
                "model": LogisticRegression(max_iter=1000),
                "params": {
                    "C": [0.1, 1, 10],
                }
            },
        }




class ModelTrainer:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.results = {}

        self.cv = StratifiedKFold(
            n_splits=Config.CV_SPLITS,
            shuffle=True,
            random_state=Config.RANDOM_STATE
        )

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y,
            test_size=Config.TEST_SIZE,
            stratify=y,
            random_state=Config.RANDOM_STATE
        )

    def train_all(self, models):
        for name, config in models.items():
            print(f"\n🔍 Training: {name}")

            grid = GridSearchCV(
                config["model"],
                config["params"],
                cv=self.cv,
                scoring="accuracy",
                n_jobs=-1
            )

            grid.fit(self.X_train, self.y_train)

            best_model = grid.best_estimator_

            y_pred = best_model.predict(self.X_test)
            y_prob = best_model.predict_proba(self.X_test)[:, 1]

            self.results[name] = {
                "model": best_model,
                "accuracy": accuracy_score(self.y_test, y_pred),
                "auc": roc_auc_score(self.y_test, y_prob),
                "params": grid.best_params_,
                "y_pred": y_pred
            }

            print(f"   Accuracy: {self.results[name]['accuracy']:.4f}")

    def get_best_model(self):
        best_name = max(self.results, key=lambda x: self.results[x]["accuracy"])
        return best_name, self.results[best_name]





class ModelPersistence:
    @staticmethod
    def save_all(model, scaler, encoder, X, y):
        # Save merged dataset
        y_df = pd.DataFrame({"Risk": [1 if i == "good" else 0 for i in y]})
        merged_df = pd.concat([X, y_df], axis=1)
        merged_df.to_csv(Config.MERGED_DATA_PATH, index=False)

        # Save artifacts
        joblib.dump(model, Config.MODEL_PATH)
        joblib.dump(scaler, Config.SCALER_PATH)

        with open(Config.ENCODER_PATH, "wb") as f:
            pickle.dump(encoder, f)

        print("\n💾 Model and artifacts saved successfully!")





class Predictor:
    def __init__(self):
        self.model = joblib.load(Config.MODEL_PATH)
        self.scaler = joblib.load(Config.SCALER_PATH)

        with open(Config.ENCODER_PATH, "rb") as f:
            self.encoder = pickle.load(f)

        self.cat_cols = self.encoder.feature_names_in_

    def predict(self, user_input):
        input_df = pd.DataFrame([user_input])

        encoded = self.encoder.transform(input_df[self.cat_cols])
        encoded_df = pd.DataFrame(
            encoded,
            columns=self.encoder.get_feature_names_out(self.cat_cols)
        )

        final_input = pd.concat(
            [input_df.drop(self.cat_cols, axis=1).reset_index(drop=True), encoded_df],
            axis=1
        )

        scaled_input = self.scaler.transform(final_input)

        prediction = self.model.predict(scaled_input)[0]

        return "Good Credit Risk ✅" if prediction == 1 else "Bad Credit Risk ⚠️"





def main():
    print("🚀 Starting End-to-End Pipeline...")

    # Step 1: Data
    processor = DataProcessor(Config.DATA_PATH)
    df = processor.load_data()
    X, y = processor.preprocess(df)

    # Step 2: Train
    trainer = ModelTrainer(X, y)
    models = ModelFactory.get_models()
    trainer.train_all(models)

    best_name, best_result = trainer.get_best_model()

    print(f"\n🏆 Best Model: {best_name}")

    # Step 3: Save
    ModelPersistence.save_all(
        best_result["model"],
        processor.scaler,
        processor.encoder,
        X,
        y
    )

    # Step 4: Test Inference
    predictor = Predictor()

    sample_input = {
        'Age': 25,
        'Sex': 'male',
        'Job': 2,
        'Housing': 'own',
        'Saving accounts': 'little',
        'Checking account': 'moderate',
        'Credit amount': 1500,
        'Duration': 12,
        'Purpose': 'radio/TV'
    }

    result = predictor.predict(sample_input)

    print("\n🧪 Sample Prediction:")
    print(result)





if __name__ == "__main__":
    main()